# 07 · Training modification M3 (mini-setup)

Assignment 1. The author's Whole model (EfficientNet-B0, his settings) trained in a small, fixed setup with **one**
change: **SNR-aware loss on the decoded signal (weight 0.1) added to the author's BCE (idea from the 11th place solution)**.

Mini-setup, the same for every run: train on folds 1-2 (3,519 images), validate on all of fold 0 (1,764 images),
6 epochs, seed 0, one GPU, batch size 4. It matches the Colab mini-baseline (notebook 06), so the validation SNR of
this run can be compared with it directly.

In [ ]:
RUN = "M3"
OVERRIDES = ["model.snr_loss_weight=0.1"]
EPOCHS = 6
REPO = "https://github.com/darren1sheehan/physionet-image-digitization-Assignment-1.git"
COMMIT = "5882869"
OUT = f"/kaggle/working/outputs/whole_b0_{RUN.lower()}"

!git clone -q $REPO /kaggle/working/repo && cd /kaggle/working/repo && git checkout -q $COMMIT && git log --oneline -1
!pip install -q "timm<1.0.29" "segmentation-models-pytorch==0.5.0" "albumentations==2.0.8" omegaconf
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Locate the attached inputs, build the mini fold CSV and link the validation ground truth
import glob, os
from pathlib import Path
import pandas as pd

def find_dir(name):
    for depth in ("*", "*/*", "*/*/*", "*/*/*/*"):
        hits = [p for p in glob.glob(f"/kaggle/input/{depth}/{name}") if os.path.isdir(p)]
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(name)

rect = {f: find_dir(f"rectified_fold{f}") for f in (0, 1, 2)}
mask_dir = find_dir("mask_x5000_500hz_comp_synthesis")
comp_train = find_dir("train")

os.chdir("/kaggle/working/repo")
df = pd.read_csv("train_fold_with_synthesis.csv", dtype={"id": str, "type_id": str})
mini = df[~df["is_synthesis"] & df["fold"].isin([0, 1, 2])].copy()
mini["image_path"] = [str(rect[f] / Path(p).name) for f, p in zip(mini["fold"], mini["image_path"])]
assert mini["image_path"].map(os.path.exists).all()
mini.to_csv("/kaggle/working/mini_folds.csv", index=False)

ecg_dir = Path("/kaggle/working/ecg_csv"); ecg_dir.mkdir(exist_ok=True)
for i in mini.loc[mini["fold"] == 0, "id"].unique():
    if not (ecg_dir / f"{i}.csv").exists():
        (ecg_dir / f"{i}.csv").symlink_to(comp_train / i / f"{i}.csv")
print("train:", (mini["fold"] != 0).sum(), " val:", (mini["fold"] == 0).sum())

In [ ]:
# Train on one GPU, as in the Colab mini-baseline
os.makedirs(OUT, exist_ok=True)
args = ["--config", "configs/whole_model.yaml", "--output_dir", OUT,
        "data.csv_path=/kaggle/working/mini_folds.csv", f"data.mask_dir={mask_dir}", f"data.ecg_dir={ecg_dir}",
        "cv.val_fold=0", "wandb.enabled=false", f"training.num_epochs={EPOCHS}", "training.max_hours=11.0",
        "num_workers=4", *OVERRIDES]
cmd = " ".join(["CUDA_VISIBLE_DEVICES=0", "python", "train.py"] + args)
print(cmd)
!{cmd}

In [ ]:
# Curves and summary
import matplotlib.pyplot as plt
hist = pd.read_csv(f"{OUT}/history.csv")
print(RUN, OVERRIDES); print(hist.to_string(index=False))
print(f"best val SNR: {hist['val_snr'].max():.2f} dB (epoch {int(hist.loc[hist['val_snr'].idxmax(), 'epoch'])})")
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist["epoch"], hist["train_loss"], "o-", label="train"); ax[0].plot(hist["epoch"], hist["val_loss"], "o-", label="val")
ax[0].set(xlabel="epoch", ylabel="loss", title=f"{RUN}: loss"); ax[0].legend()
ax[1].plot(hist["epoch"], hist["val_snr"], "o-"); ax[1].set(xlabel="epoch", ylabel="dB", title=f"{RUN}: validation SNR (fold 0)")
plt.tight_layout(); plt.savefig(f"{OUT}/curves.png", dpi=120); plt.show()